<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 40px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white;">
  <span style="background: rgba(255,255,255,0.2); border: 1px solid rgba(255,255,255,0.4); color: white; padding: 4px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; text-transform: uppercase;">Kafka Training · Lab 10</span>
  <h1 style="color: #ffffff; font-size: 2.4em; font-weight: bold; margin-top: 15px;">Kafka Connect with JDBC Source Connector</h1>
  <p style="color: #e0e0e0; font-size: 1.1em;">Learn how to stream data from PostgreSQL into Kafka using the JDBC source connector.</p>
</div>

---

## 🎯 Overview

**Kafka Connect** is a framework for building scalable and reliable data pipelines between Kafka and external systems. It enables:
- **Source Connectors**: Stream data FROM external systems INTO Kafka
- **Sink Connectors**: Write data FROM Kafka TO external systems

**In this lab, we'll:**
1. Run PostgreSQL in Docker
2. Create sample customer data in PostgreSQL
3. Deploy a JDBC source connector to stream data to Kafka
4. Consume and verify the transformed data

---




---

## <span style="color: #667eea;">Step 1:</span> Create Enhanced Docker Compose for Kafka Connect + PostgreSQL

First, let's create a comprehensive docker-compose.yml that includes:
- **Zookeeper**: Coordination service
- **Kafka Broker**: Message broker (with JDBC plugin)
- **Kafka Connect**: Distributed connector runtime
- **PostgreSQL**: Sample database
- **Control Center** (optional): UI for monitoring

Create a new docker-compose.yml in Lab10 directory with all services:



In [ ]:
!docker-compose -f ../../docker-compose.yml down
!docker-compose down -v
import time
time.sleep(5)

In [ ]:
!docker-compose up -d --build
import time
time.sleep(5)

In [ ]:
!docker ps

### Install psycopg2 (PostgreSQL client)

In [ ]:
!pip install psycopg2-binary

### Create PostgreSQL Table and Insert Sample Data

In [ ]:
import psycopg2
import time

# Wait for PostgreSQL to be ready
print("⏳ Waiting for PostgreSQL to be ready...")
for attempt in range(30):
    try:
        conn = psycopg2.connect(
            host='localhost',
            port=5432,
            database='demo_db',
            user='demo_user',
            password='demo_password'
        )
        conn.close()
        print("✓ PostgreSQL is ready!")
        break
    except psycopg2.OperationalError:
        if attempt == 29:
            raise
        time.sleep(1)

# Connect and create table
conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='demo_db',
    user='demo_user',
    password='demo_password'
)
cursor = conn.cursor()

# Create customers table with incrementing ID
sql_create_table = """
DROP TABLE IF EXISTS customers;
CREATE TABLE customers (
    id SERIAL PRIMARY KEY,
    customer_name VARCHAR(100) NOT NULL,
    email VARCHAR(100) NOT NULL,
    phone VARCHAR(20),
    country VARCHAR(50),
    registration_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    annual_revenue DECIMAL(10, 2)
);

-- Insert sample data
INSERT INTO customers (customer_name, email, phone, country, annual_revenue) VALUES
('Acme Corp', 'contact@acme.com', '+1-555-0101', 'USA', 5000000.00),
('Global Enterprises', 'info@globalent.com', '+44-20-7946-0958', 'UK', 8500000.00),
('Tech Solutions Ltd', 'sales@techsol.com', '+33-1-42-68-53-00', 'France', 3200000.00),
('Innovation Hub', 'hello@innovation.com', '+49-30-1234567', 'Germany', 4100000.00),
('Digital First Co', 'contact@digitalfirst.com', '+81-3-1234-5678', 'Japan', 6750000.00);
"""

cursor.execute(sql_create_table)
conn.commit()

# Verify data
cursor.execute("SELECT COUNT(*) FROM customers;")
count = cursor.fetchone()[0]
print(f"✓ Created customers table with {count} records")

cursor.execute("SELECT id, customer_name, email, country FROM customers LIMIT 3;")
print("\n📋 Sample data from customers table:")
for row in cursor.fetchall():
    print(f"  ID: {row[0]}, Name: {row[1]}, Email: {row[2]}, Country: {row[3]}")

cursor.close()
conn.close()


---

## <span style="color: #667eea;">Step 4:</span> JDBC Source Connector Configuration

In [ ]:
import json
import time

# JDBC Source Connector configuration with SMT
connector_config = {
    "name": "postgres-customers-connector",
    "config": {
        # Connection settings
        "connector.class": "io.confluent.connect.jdbc.JdbcSourceConnector",
        "connection.url": "jdbc:postgresql://postgres:5432/demo_db",
        "connection.user": "demo_user",
        "connection.password": "demo_password",
        
        # Polling and tracking
        "mode": "incrementing",
        "incrementing.column.name": "id",
        "poll.interval.ms": "60000",
        "batch.max.rows": "100",
        
        # Topic and data mapping
        "table.whitelist": "customers",
        "topic.prefix": "lab10-",
        
        # Schema handling
        "quote.sql.identifiers": "always",
        "numeric.precision.mapping": "true",
        
        # Task settings
        "tasks.max": "1"
    }
}

# Save connector configuration to JSON file
with open('connector_config.json', 'w') as f:
    json.dump(connector_config, f)

print("✓ Connector configuration saved to connector_config.json")


## <span style="color: #667eea;">Step 5:</span> Deploy Connecto5

In [ ]:
print("📤 Deploying JDBC Source Connector...\n")
!curl -s -X POST -H "Content-Type: application/json" -d @connector_config.json http://localhost:8083/connectors
print("\n✓ Connector deployment request submitted!")


---

## <span style="color: #667eea;">Step 6:</span> Monitor Connector Status

Check the status of the deployed JDBC connector using the Kafka Connect REST API:

### Check Connector Status



In [ ]:
!curl -s http://localhost:8083/ 


In [ ]:
print("\n📋 List all connectors:\n")
!curl -s http://localhost:8083/connectors


In [ ]:
print("\n📊 Connector Status:\n")
!curl -s http://localhost:8083/connectors/postgres-customers-connector/status


---

## <span style="color: #667eea;">Step 7:</span> Verify Data in Kafka Topic

Wait for the connector to poll and ingest data, then consume messages from the topic to verify the transformation:



In [ ]:
!docker exec lab10-kafka-1 kafka-console-consumer --bootstrap-server localhost:9092 --topic lab10-customers --from-beginning --max-messages 5 --property print.key=true --property key.separator=" | " --timeout-ms 10000



---

<div style="background: linear-gradient(135deg, #38a169 0%, #276749 100%); padding: 30px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white; margin-top: 40px;">
  <h3 style="color: #ffffff; margin-top: 0;">🎉 Lab 10 Complete!</h3>
  <p style="color: #d1fae5; margin: 15px 0;">You've successfully mastered Kafka Connect with JDBC Source Connector and Single Message Transforms!</p>
  
</div>
